# Fixed Hospital Split: Feature Selection vs Cross-Hospital Generalization

## Question
Holding the **outcome, hospital split, model, preprocessing, and metric fixed**, how does the
**feature-selection method** change cross-hospital generalization?

Protocol:
1. Fix source hospitals and held-out hospitals using a hospital attribute.
2. Split source-hospital patient rows 80/20, keeping the same hospitals in both subsets.
3. Select features using the 80% source-train rows only.
4. Train the same logistic-regression model for every feature set.
5. Evaluate internal AUROC on the 20% source validation rows and AUROC separately at each held-out hospital.
6. Define `gap = internal AUROC - held-out hospital AUROC`.
7. Compare the gap distribution across feature-selection methods.

### Primary defaults
- **Outcome:** `label_J96.00` (future respiratory failure after the first 24-hour feature window).
- **Source hospitals:** Midwest.
- **Held-out hospitals:** Northeast hospitals meeting minimum sample-size and class-count thresholds.
- **Primary metric:** AUROC; AUPRC and Brier are secondary.

These defaults are based on prior full-data work: `label_J96.00` gave clearer hospital-to-hospital
differences than mortality, and region-based transfer produced meaningful gaps. The split is fixed
before comparing feature selectors.

### Feature-selection methods
1. All eligible features (control).
2. Top pooled predictive utility.
3. L1 logistic-regression ranking.
4. Cross-site consensus (OHDSI-inspired proxy).
5. StableMate-inspired predictive-then-stable ranking.
6. Project score: `utility - lambda * instability`.

The OHDSI- and StableMate-inspired methods are **proxies**, not exact reproductions.

In [ ]:
from pathlib import Path
import pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
SEED = 42
rng = np.random.default_rng(SEED)

In [ ]:
# Primary experiment
OUTCOME = "label_J96.00"
TRAIN_REGION = "Midwest"
TEST_REGION = "Northeast"
INTERNAL_VALIDATION_FRACTION = 0.20

# Held-out hospital eligibility
MIN_TEST_HOSPITAL_N = 300
MIN_TEST_POSITIVES = 15
MIN_TEST_NEGATIVES = 15

# Source hospitals used for stability calculations
MIN_SOURCE_ENV_N = 150
MIN_SOURCE_ENV_POSITIVES = 5
MIN_SOURCE_ENV_NEGATIVES = 5

# Source-only candidate filtering
MIN_FEATURE_NONMISSING_RATE = 0.05
MIN_FEATURE_NONMISSING_COUNT = 500
MIN_SOURCE_HOSPITALS_WITH_FEATURE = 5
MIN_OBS_PER_HOSPITAL_FOR_FEATURE = 30
MAX_FEATURES_PER_GROUP = 40
MAX_TOTAL_CANDIDATES = 160

# Keep feature count fixed for selector comparisons
N_SELECTED = 30

L1_C = 0.05
STABLEMATE_PREDICTIVE_POOL = 60
UTILITY_INSTABILITY_LAMBDA = 0.50
FINAL_MODEL_C = 0.10

OUTPUT_DIR = Path("feature_selection_generalization_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load the full processed eICU data

In [ ]:
def find_project_data(filename):
    cwd = Path.cwd().resolve()
    candidates = []
    for base in [cwd, *list(cwd.parents)[:6]]:
        candidates.append(base / "data" / filename)
    candidates += [
        (cwd / "../../data" / filename).resolve(),
        (cwd / "../../../data" / filename).resolve(),
    ]
    seen = set()
    for p in candidates:
        p = p.resolve()
        if p in seen:
            continue
        seen.add(p)
        if p.exists():
            return p
    tried = "\n".join(f"  - {p}" for p in seen)
    raise FileNotFoundError(f"Could not find {filename}. Tried:\n{tried}")

DATA_PATH = find_project_data("clean_dataset.parquet")
FEATURE_NAMES_PATH = find_project_data("feature_names.pkl")

feature_sets = pickle.load(open(FEATURE_NAMES_PATH, "rb"))
patient_agg = pd.read_parquet(DATA_PATH)

if "age" in patient_agg.columns:
    patient_agg.loc[patient_agg["age"] == "> 89", "age"] = "90"
    patient_agg["age"] = pd.to_numeric(patient_agg["age"], errors="coerce")

if OUTCOME not in patient_agg.columns:
    raise KeyError(f"{OUTCOME!r} is not in the dataset.")

patient_agg[OUTCOME] = patient_agg[OUTCOME].fillna(0).astype(int)

print("Data:", DATA_PATH)
print("Shape:", patient_agg.shape)
print(f"{OUTCOME} prevalence: {patient_agg[OUTCOME].mean():.3%}")

## 2. Define candidate features and lock the hospital split

In [ ]:
labs = [c for c in feature_sets["labs"].tolist() if c in patient_agg.columns]
vitals = [c for c in feature_sets["vitals"].tolist() if c in patient_agg.columns]
icd = [c for c in feature_sets["icd10_before24h"].tolist() if c in patient_agg.columns]
admissiondx = [c for c in feature_sets["admissiondx"].tolist() if c in patient_agg.columns]
demographic_numeric = [c for c in ["age", "admissionheight", "admissionweight"] if c in patient_agg.columns]

raw_feature_group = {}
for c in labs: raw_feature_group[c] = "labs"
for c in vitals: raw_feature_group[c] = "vitals"
for c in icd: raw_feature_group[c] = "icd10_before24h"
for c in admissiondx: raw_feature_group[c] = "admissiondx"
for c in demographic_numeric: raw_feature_group[c] = "demographic"

raw_candidates = list(dict.fromkeys(labs + vitals + icd + admissiondx + demographic_numeric))
raw_candidates = [
    c for c in raw_candidates
    if "J96" not in c.upper()
    and not c.startswith("label_")
    and c not in {"hospitalid", "hospital_region", "hospital_numbedscategory"}
]

for c in raw_candidates:
    patient_agg[c] = pd.to_numeric(patient_agg[c], errors="coerce")

hospital_stats = (
    patient_agg.groupby("hospitalid")
    .agg(
        n=("hospitalid", "size"),
        positives=(OUTCOME, "sum"),
        prevalence=(OUTCOME, "mean"),
        region=("hospital_region", "first"),
    )
)
hospital_stats["negatives"] = hospital_stats["n"] - hospital_stats["positives"]

train_hospital_ids = hospital_stats.index[hospital_stats["region"].eq(TRAIN_REGION)].tolist()
test_hospital_ids = hospital_stats.index[
    hospital_stats["region"].eq(TEST_REGION)
    & (hospital_stats["n"] >= MIN_TEST_HOSPITAL_N)
    & (hospital_stats["positives"] >= MIN_TEST_POSITIVES)
    & (hospital_stats["negatives"] >= MIN_TEST_NEGATIVES)
].tolist()

if len(train_hospital_ids) < 2 or len(test_hospital_ids) < 2:
    raise ValueError("Too few hospitals for the configured regional split.")

source_all = patient_agg[patient_agg["hospitalid"].isin(train_hospital_ids)].copy()
test_all = patient_agg[patient_agg["hospitalid"].isin(test_hospital_ids)].copy()

study_hospitals = len(train_hospital_ids) + len(test_hospital_ids)
print(f"TRAIN: {TRAIN_REGION}: {len(train_hospital_ids)} hospitals, {len(source_all):,} patients, prevalence={source_all[OUTCOME].mean():.3%}")
print(f"TEST:  {TEST_REGION}: {len(test_hospital_ids)} eligible hospitals, {len(test_all):,} patients, prevalence={test_all[OUTCOME].mean():.3%}")
print(f"Hospital split in this two-region cohort: {len(train_hospital_ids)/study_hospitals:.1%} train / {len(test_hospital_ids)/study_hospitals:.1%} test")
display(hospital_stats.loc[test_hospital_ids].sort_values("n", ascending=False))

## 3. Fixed 80/20 patient split within source hospitals

In [ ]:
source_train, source_internal = train_test_split(
    source_all,
    test_size=INTERNAL_VALIDATION_FRACTION,
    random_state=SEED,
    stratify=source_all[OUTCOME],
)
source_train = source_train.copy()
source_internal = source_internal.copy()

print("source_train:", source_train.shape, "hospitals:", source_train["hospitalid"].nunique())
print("source_internal:", source_internal.shape, "hospitals:", source_internal["hospitalid"].nunique())

## 4. Source-only candidate quality filter

In [ ]:
def build_candidate_pool(df):
    rows = []
    for feature in raw_candidates:
        x = pd.to_numeric(df[feature], errors="coerce")
        nonmissing = int(x.notna().sum())
        nonmissing_rate = float(x.notna().mean())
        nunique = int(x.nunique(dropna=True))
        per_hosp_obs = x.notna().groupby(df["hospitalid"]).sum()
        hosp_coverage = int((per_hosp_obs >= MIN_OBS_PER_HOSPITAL_FOR_FEATURE).sum())

        if (
            nonmissing >= MIN_FEATURE_NONMISSING_COUNT
            and nonmissing_rate >= MIN_FEATURE_NONMISSING_RATE
            and hosp_coverage >= MIN_SOURCE_HOSPITALS_WITH_FEATURE
            and nunique > 1
        ):
            rows.append({
                "feature": feature,
                "feature_group": raw_feature_group.get(feature, "other"),
                "nonmissing": nonmissing,
                "nonmissing_rate": nonmissing_rate,
                "hospital_coverage": hosp_coverage,
                "nunique": nunique,
            })

    coverage = pd.DataFrame(rows)
    if coverage.empty:
        raise ValueError("No features survived the source-only coverage filter.")

    pieces = []
    for group, g in coverage.groupby("feature_group"):
        pieces.append(
            g.sort_values(["hospital_coverage", "nonmissing"], ascending=False)
             .head(MAX_FEATURES_PER_GROUP)
        )

    kept = pd.concat(pieces, ignore_index=True)
    kept = kept.sort_values(["hospital_coverage", "nonmissing"], ascending=False).head(MAX_TOTAL_CANDIDATES)
    return kept.reset_index(drop=True)

candidate_table = build_candidate_pool(source_train)
candidate_features = candidate_table["feature"].tolist()

print("Candidate features:", len(candidate_features))
display(candidate_table.groupby("feature_group").size().rename("n_features"))

## 5. Predictive utility and cross-hospital instability

All scores below use **source-train only**.

- Predictive utility = `2 * abs(univariate AUROC - 0.5)`.
- Missingness instability = SD of hospital-level observation rates.
- Distribution instability = leave-one-hospital-out KS shift (continuous) or prevalence shift (binary).
- Relationship instability = SD of signed per-hospital AUROC (`AUC - 0.5`).

In [ ]:
def feature_kind(series):
    vals = pd.to_numeric(series, errors="coerce").dropna().unique()
    if len(vals) <= 2 and set(np.unique(vals)).issubset({0, 1}):
        return "binary"
    return "continuous"

def pooled_utility(feature, df):
    x = pd.to_numeric(df[feature], errors="coerce")
    y = df[OUTCOME]
    mask = x.notna()
    if mask.sum() < MIN_FEATURE_NONMISSING_COUNT or y[mask].nunique() < 2:
        return np.nan, np.nan
    auc = roc_auc_score(y[mask], x[mask])
    return auc, 2 * abs(auc - 0.5)

source_env_stats = source_train.groupby("hospitalid")[OUTCOME].agg(["count", "sum"])
source_env_stats["negatives"] = source_env_stats["count"] - source_env_stats["sum"]

stability_hospitals = source_env_stats.index[
    (source_env_stats["count"] >= MIN_SOURCE_ENV_N)
    & (source_env_stats["sum"] >= MIN_SOURCE_ENV_POSITIVES)
    & (source_env_stats["negatives"] >= MIN_SOURCE_ENV_NEGATIVES)
].tolist()

print("Source hospitals usable for hospital-level stability:", len(stability_hospitals))

In [ ]:
def compute_feature_scores(df, features):
    rows = []

    for i, feature in enumerate(features, start=1):
        x = pd.to_numeric(df[feature], errors="coerce")
        y = df[OUTCOME]
        kind = feature_kind(x)
        pooled_auc, utility = pooled_utility(feature, df)

        observed_rates = x.notna().groupby(df["hospitalid"]).mean()
        miss_inst = float(observed_rates.std(ddof=0))

        dist_scores = []
        rel_effects = []

        for h in stability_hospitals:
            hmask = df["hospitalid"].eq(h)
            x_h = x[hmask].dropna()
            x_other = x[~hmask].dropna()

            if len(x_h) >= MIN_OBS_PER_HOSPITAL_FOR_FEATURE and len(x_other) >= MIN_OBS_PER_HOSPITAL_FOR_FEATURE:
                if kind == "binary":
                    dist_scores.append(abs(float(x_h.mean()) - float(x_other.mean())))
                else:
                    a, b = x_h.to_numpy(), x_other.to_numpy()
                    if len(a) > 750: a = rng.choice(a, 750, replace=False)
                    if len(b) > 3000: b = rng.choice(b, 3000, replace=False)
                    dist_scores.append(ks_2samp(a, b, mode="auto").statistic)

            pair = hmask & x.notna()
            yh = y[pair]
            xh = x[pair]
            if (
                len(xh) >= MIN_SOURCE_ENV_N
                and yh.nunique() == 2
                and yh.sum() >= MIN_SOURCE_ENV_POSITIVES
                and (len(yh) - yh.sum()) >= MIN_SOURCE_ENV_NEGATIVES
            ):
                rel_effects.append(roc_auc_score(yh, xh) - 0.5)

        dist_inst = float(np.mean(dist_scores)) if len(dist_scores) >= MIN_SOURCE_HOSPITALS_WITH_FEATURE else np.nan
        rel_inst = float(np.std(rel_effects, ddof=0)) if len(rel_effects) >= MIN_SOURCE_HOSPITALS_WITH_FEATURE else np.nan

        rows.append({
            "feature": feature,
            "feature_group": raw_feature_group.get(feature, "other"),
            "predictive_utility": utility,
            "pooled_auc": pooled_auc,
            "missingness_instability": miss_inst,
            "distribution_instability": dist_inst,
            "relationship_instability": rel_inst,
            "n_relationship_hospitals": len(rel_effects),
        })

        if i % 25 == 0 or i == len(features):
            print(f"Scored {i}/{len(features)}")

    scores = pd.DataFrame(rows)

    for col in ["predictive_utility", "missingness_instability", "distribution_instability", "relationship_instability"]:
        scores[col + "_rank"] = scores[col].rank(pct=True, method="average")

    inst_rank_cols = [
        "missingness_instability_rank",
        "distribution_instability_rank",
        "relationship_instability_rank",
    ]
    complete = scores[inst_rank_cols].notna().all(axis=1)
    scores["combined_instability"] = np.nan
    scores.loc[complete, "combined_instability"] = scores.loc[complete, inst_rank_cols].mean(axis=1)

    valid = scores["predictive_utility_rank"].notna() & scores["combined_instability"].notna()
    scores["utility_instability_score"] = np.nan
    scores.loc[valid, "utility_instability_score"] = (
        scores.loc[valid, "predictive_utility_rank"]
        - UTILITY_INSTABILITY_LAMBDA * scores.loc[valid, "combined_instability"]
    )
    return scores

feature_scores = compute_feature_scores(source_train, candidate_features)
display(feature_scores.sort_values("predictive_utility", ascending=False).head(15))

## 6. Feature-selection methods

In [ ]:
def select_top_utility(scores, k):
    return (
        scores.dropna(subset=["predictive_utility"])
        .sort_values("predictive_utility", ascending=False)
        .head(k)["feature"].tolist()
    )

def select_l1(df, features, k):
    X = df[features].apply(pd.to_numeric, errors="coerce")
    y = df[OUTCOME].astype(int)
    imp = SimpleImputer(strategy="median")
    scale = StandardScaler()
    Xs = scale.fit_transform(imp.fit_transform(X))
    m = LogisticRegression(
        penalty="l1", solver="liblinear", C=L1_C,
        class_weight="balanced", max_iter=2000, random_state=SEED
    ).fit(Xs, y)
    ranking = pd.Series(np.abs(m.coef_[0]), index=features, name="abs_l1_coef").sort_values(ascending=False)
    return ranking.head(k).index.tolist(), ranking

def build_cross_site_consensus(df, features, k):
    counts = pd.Series(0.0, index=features)
    used = 0
    for h in stability_hospitals:
        hdf = df[df["hospitalid"].eq(h)]
        y = hdf[OUTCOME]
        if y.nunique() != 2:
            continue
        vals = []
        for feature in features:
            x = pd.to_numeric(hdf[feature], errors="coerce")
            mask = x.notna()
            if mask.sum() < MIN_OBS_PER_HOSPITAL_FOR_FEATURE or y[mask].nunique() != 2:
                continue
            auc = roc_auc_score(y[mask], x[mask])
            vals.append((feature, 2 * abs(auc - 0.5)))
        if not vals:
            continue
        used += 1
        for feature, _ in sorted(vals, key=lambda t: t[1], reverse=True)[:k]:
            counts.loc[feature] += 1

    if used == 0:
        raise ValueError("No source hospitals usable for consensus selector.")

    consensus = (counts / used).rename("consensus_frequency")
    pooled = feature_scores.set_index("feature")["predictive_utility"]
    ranking = pd.concat([consensus, pooled], axis=1).sort_values(
        ["consensus_frequency", "predictive_utility"], ascending=False
    )
    return ranking.head(k).index.tolist(), ranking

def select_stablemate_proxy(scores, k):
    pool = (
        scores.dropna(subset=["predictive_utility", "relationship_instability"])
        .sort_values("predictive_utility", ascending=False)
        .head(min(STABLEMATE_PREDICTIVE_POOL, len(scores)))
        .copy()
    )
    pool = pool.sort_values(["relationship_instability", "predictive_utility"], ascending=[True, False])
    return pool.head(k)["feature"].tolist(), pool

def select_utility_instability(scores, k):
    ranking = scores.dropna(subset=["utility_instability_score"]).sort_values(
        "utility_instability_score", ascending=False
    )
    return ranking.head(k)["feature"].tolist(), ranking

top_utility = select_top_utility(feature_scores, N_SELECTED)
l1_features, l1_ranking = select_l1(source_train, candidate_features, N_SELECTED)
consensus_features, consensus_ranking = build_cross_site_consensus(source_train, candidate_features, N_SELECTED)
stablemate_features, stablemate_ranking = select_stablemate_proxy(feature_scores, N_SELECTED)
utility_instability_features, utility_instability_ranking = select_utility_instability(feature_scores, N_SELECTED)

selected_feature_sets = {
    "all_eligible": candidate_features,
    f"top{N_SELECTED}_utility": top_utility,
    f"top{N_SELECTED}_l1": l1_features,
    f"top{N_SELECTED}_cross_site_consensus": consensus_features,
    f"top{N_SELECTED}_stablemate_proxy": stablemate_features,
    f"top{N_SELECTED}_utility_minus_instability": utility_instability_features,
}

for name, feats in selected_feature_sets.items():
    print(f"{name:40s} {len(feats):3d}")

In [ ]:
selection_rows = []
for method, feats in selected_feature_sets.items():
    for rank, feature in enumerate(feats, start=1):
        selection_rows.append({
            "method": method,
            "rank": rank,
            "feature": feature,
            "feature_group": raw_feature_group.get(feature, "other"),
        })

selection_table = pd.DataFrame(selection_rows)
group_counts = selection_table.groupby(["method", "feature_group"]).size().unstack(fill_value=0)
display(group_counts)

selection_table.to_csv(OUTPUT_DIR / "selected_features_by_method.csv", index=False)
feature_scores.to_csv(OUTPUT_DIR / "source_feature_scores.csv", index=False)

## 7. Train the identical final model for every selected feature set

In [ ]:
def make_final_model():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler()),
        ("model", LogisticRegression(
            C=FINAL_MODEL_C,
            max_iter=2500,
            class_weight="balanced",
            random_state=SEED,
        )),
    ])

def evaluate_method(method, features):
    model = make_final_model()
    Xtr = source_train[features].apply(pd.to_numeric, errors="coerce")
    ytr = source_train[OUTCOME].astype(int)
    Xint = source_internal[features].apply(pd.to_numeric, errors="coerce")
    yint = source_internal[OUTCOME].astype(int)

    model.fit(Xtr, ytr)
    pint = model.predict_proba(Xint)[:, 1]
    int_auc = roc_auc_score(yint, pint)
    int_pr = average_precision_score(yint, pint)
    int_brier = brier_score_loss(yint, pint)

    rows = [{
        "method": method, "evaluation": "internal", "hospitalid": "internal",
        "n": len(yint), "positives": int(yint.sum()), "prevalence": float(yint.mean()),
        "auroc": int_auc, "auprc": int_pr, "brier": int_brier,
        "gap_auroc": 0.0, "n_features": len(features),
    }]

    for h in sorted(test_hospital_ids):
        hdf = test_all[test_all["hospitalid"].eq(h)]
        yh = hdf[OUTCOME].astype(int)
        if yh.nunique() != 2:
            continue
        Xh = hdf[features].apply(pd.to_numeric, errors="coerce")
        ph = model.predict_proba(Xh)[:, 1]
        auc = roc_auc_score(yh, ph)
        rows.append({
            "method": method, "evaluation": "heldout", "hospitalid": h,
            "n": len(yh), "positives": int(yh.sum()), "prevalence": float(yh.mean()),
            "auroc": auc,
            "auprc": average_precision_score(yh, ph),
            "brier": brier_score_loss(yh, ph),
            "gap_auroc": int_auc - auc,
            "n_features": len(features),
        })
    return pd.DataFrame(rows), model

frames, fitted_models = [], {}
for method, features in selected_feature_sets.items():
    print("Fitting", method)
    perf, model = evaluate_method(method, features)
    frames.append(perf)
    fitted_models[method] = model

performance = pd.concat(frames, ignore_index=True)
performance.to_csv(OUTPUT_DIR / "hospital_level_performance.csv", index=False)
display(performance.head(20))

## 8. Primary result: generalization-gap comparison

In [ ]:
heldout_perf = performance[performance["evaluation"].eq("heldout")].copy()

summary = (
    heldout_perf.groupby("method")
    .agg(
        n_hospitals=("hospitalid", "nunique"),
        n_features=("n_features", "first"),
        mean_gap=("gap_auroc", "mean"),
        median_gap=("gap_auroc", "median"),
        worst_gap=("gap_auroc", "max"),
        median_heldout_auroc=("auroc", "median"),
        mean_heldout_auroc=("auroc", "mean"),
        median_heldout_auprc=("auprc", "median"),
    )
)

internal = (
    performance[performance["evaluation"].eq("internal")]
    .set_index("method")[["auroc", "auprc", "brier"]]
    .rename(columns={"auroc":"internal_auroc","auprc":"internal_auprc","brier":"internal_brier"})
)
summary = summary.join(internal).sort_values("median_gap")
summary.to_csv(OUTPUT_DIR / "method_summary.csv")
display(summary.round(4))

In [ ]:
order = summary.index.tolist()
data = [heldout_perf.loc[heldout_perf["method"].eq(m), "gap_auroc"].to_numpy() for m in order]

plt.figure(figsize=(11, 6))
plt.boxplot(data, labels=order, showmeans=True)
plt.axhline(0, linewidth=1)
plt.ylabel("Generalization gap: internal AUROC - held-out AUROC")
plt.xlabel("Feature-selection method")
plt.title(f"{OUTCOME}: {TRAIN_REGION} source -> {TEST_REGION} held-out hospitals")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "generalization_gap_by_method.png", dpi=220, bbox_inches="tight")
plt.show()

In [ ]:
gap_matrix = heldout_perf.pivot(index="hospitalid", columns="method", values="gap_auroc")

plt.figure(figsize=(12, max(4, 0.45 * len(gap_matrix))))
im = plt.imshow(gap_matrix[order].to_numpy(), aspect="auto")
plt.colorbar(im, label="Internal AUROC - held-out AUROC")
plt.yticks(range(len(gap_matrix)), gap_matrix.index.astype(str))
plt.xticks(range(len(order)), order, rotation=35, ha="right")
plt.xlabel("Feature-selection method")
plt.ylabel("Held-out hospital")
plt.title("Hospital-by-method generalization gaps")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "gap_heatmap_hospital_by_method.png", dpi=220, bbox_inches="tight")
plt.show()

## 9. Interpretability: utility vs instability and internal vs external tradeoff

In [ ]:
plot_scores = feature_scores.dropna(subset=["predictive_utility", "combined_instability"]).copy()

plt.figure(figsize=(9, 7))
plt.scatter(plot_scores["combined_instability"], plot_scores["predictive_utility"], alpha=0.55)
plot_scores["interesting"] = plot_scores["predictive_utility_rank"] + plot_scores["combined_instability"]
for _, row in plot_scores.sort_values("interesting", ascending=False).head(15).iterrows():
    plt.annotate(row["feature"], (row["combined_instability"], row["predictive_utility"]), fontsize=7)
plt.xlabel("Combined source-hospital instability rank")
plt.ylabel("Pooled source predictive utility")
plt.title(f"{OUTCOME}: predictive utility vs instability")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "utility_vs_instability.png", dpi=220, bbox_inches="tight")
plt.show()

tradeoff = summary.reset_index()
plt.figure(figsize=(8, 6))
plt.scatter(tradeoff["internal_auroc"], tradeoff["median_heldout_auroc"])
for _, row in tradeoff.iterrows():
    plt.annotate(row["method"], (row["internal_auroc"], row["median_heldout_auroc"]), fontsize=8)
lims = [
    min(tradeoff["internal_auroc"].min(), tradeoff["median_heldout_auroc"].min()) - 0.01,
    max(tradeoff["internal_auroc"].max(), tradeoff["median_heldout_auroc"].max()) + 0.01,
]
plt.plot(lims, lims, linestyle="--", linewidth=1)
plt.xlim(lims); plt.ylim(lims)
plt.xlabel("Internal AUROC")
plt.ylabel("Median held-out AUROC")
plt.title("Internal performance vs cross-hospital performance")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "internal_vs_heldout_auroc.png", dpi=220, bbox_inches="tight")
plt.show()

## 10. What to conclude before changing the setup

Answer these before trying another split or outcome:

1. Which selector has the smallest median/mean gap?
2. Did it preserve held-out AUROC, or did it merely lower internal AUROC?
3. Do the same hospitals remain difficult across selectors?
4. Which features are selected by the best method but not by pooled utility?
5. Which highly predictive features are removed by stability-aware methods?
6. Are labs/vitals disproportionately removed relative to diagnosis features?
7. Does the source-only utility-vs-instability plot explain the model-level result?

### Robustness checks only after the primary result is recorded
- outcome: `mortality_at48h`;
- another regional transfer;
- another `N_SELECTED`;
- another model class;
- another stability penalty `lambda`.

Do not redefine the primary split after seeing which setup makes a selector look best.

In [ ]:
print("Outputs saved to:", OUTPUT_DIR.resolve())
display(summary.round(4))
display(group_counts)